# 06. Ciekawostki dla dociekliwych

Ten notebook jest **dodatkowy**: pokazuje narzędzia i techniki, które uzupełniają testy jednostkowe
i często znajdują błędy, których zwykłe testy z przykładowymi danymi nie złapią.
Wszystkie demonstracje są w katalogu [`demos/`](demos/CMakeLists.txt).

* sanitizery — testy, które wykrywają błędy pamięci i zachowanie niezdefiniowane,
* death tests — testowanie, że program się zatrzymuje,
* `static_assert` — testy wykonywane przez kompilator,
* fuzzing i testy oparte na właściwościach — testy z losowymi danymi,
* kata TDD i inne frameworki testowe.

## Sanitizery: ASan i UBSan

W C++ błąd pamięci (zapis poza tablicą, użycie zwolnionej pamięci) albo zachowanie niezdefiniowane
(np. przepełnienie `int`) wcale nie musi spowodować widocznego błędu. Program może działać poprawnie
„przez przypadek”, a testy mogą przechodzić. **Sanitizery** to instrumentacja dodawana przez kompilator,
która wykrywa takie błędy w trakcie działania programu:

* **AddressSanitizer (ASan)** — wyjście poza zakres (sterta, stos, zmienne globalne), użycie po zwolnieniu,
  podwójne zwolnienie, wycieki; spowalnia program typowo ok. 2 razy
  ([Clang — AddressSanitizer](https://clang.llvm.org/docs/AddressSanitizer.html)),
* **UndefinedBehaviorSanitizer (UBSan)** — m.in. przepełnienie liczb ze znakiem, przesunięcia bitowe
  poza zakres, dzielenie przez zero
  ([Clang — UBSan](https://clang.llvm.org/docs/UndefinedBehaviorSanitizer.html)).

Włącza się je flagami kompilacji i linkowania: `-fsanitize=address,undefined` (GCC i Clang;
[GCC](https://gcc.gnu.org/onlinedocs/gcc/Instrumentation-Options.html#index-fsanitize_003daddress)),
`/fsanitize=address` w MSVC. Domyślnie UBSan tylko wypisuje ostrzeżenie i działa dalej — opcja
`-fno-sanitize-recover=all` sprawia, że program kończy się błędem, więc test nie przechodzi.
Uruchamianie całego zestawu testów z sanitizerami (np. w CI) to tani sposób na znalezienie błędów,
których żadna asercja nie sprawdza.

Plik [`demos/sanitizer_test.cpp`](demos/sanitizer_test.cpp) zawiera funkcję, która zapisuje jeden element
za końcem wektora, i test, który sprawdza tylko poprawne elementy. Najpierw zwykła kompilacja wszystkich
demonstracji:

In [ ]:
# Normal build: every test passes - including the one with a memory bug
!cmake -S demos -B demos/build && cmake --build demos/build --config Debug --parallel && ctest --test-dir demos/build -C Debug --output-on-failure

Teraz ten sam kod zbudowany z ASan i UBSan (osobny katalog `build-san`). Szukaj w wyniku linii
`ERROR: AddressSanitizer: heap-buffer-overflow` i numeru linii w `sanitizer_test.cpp`.

In [ ]:
# Sanitizer build (GCC/Clang or MSVC): the memory bug is reported and the test fails
!cmake -S demos -B demos/build-san -DENABLE_SANITIZERS=ON && cmake --build demos/build-san --config Debug --parallel && ctest --test-dir demos/build-san -C Debug --output-on-failure

Materiały: [K. Serebryany — *Sanitize your C++ code*, CppCon 2014 (wideo)](https://www.youtube.com/watch?v=V2_80g0eOMc),
[V. Ciura — *2020: The Year of Sanitizers?*, CppCon 2020 (wideo, m.in. MSVC)](https://www.youtube.com/watch?v=xEzfnbTabyE),
[Trail of Bits — *Understanding AddressSanitizer*](https://blog.trailofbits.com/2024/05/16/understanding-addresssanitizer-better-memory-safety-for-your-code/).

## Death tests

Czasem poprawnym zachowaniem jest **zatrzymanie programu** — np. funkcja sprawdza warunek, którego
złamanie oznacza błąd programisty, i wywołuje `std::abort()`. GoogleTest potrafi to przetestować
([Advanced — Death Tests](https://google.github.io/googletest/advanced.html#death-tests)):

* `EXPECT_DEATH(instrukcja, regex)` — instrukcja kończy proces z niezerowym kodem, a na `stderr`
  pojawia się tekst pasujący do wyrażenia regularnego,
* `EXPECT_EXIT(instrukcja, predykat, regex)` — np. `::testing::ExitedWithCode(0)`.

Instrukcja jest wykonywana w osobnym procesie potomnym, więc test „przeżywa” jego zakończenie.
Zestawy z death testami nazywa się z końcówką `DeathTest` — GoogleTest uruchamia je wtedy przed
pozostałymi ([Death Test Naming](https://google.github.io/googletest/advanced.html#death-test-naming)).
Przykład: [`demos/death_test.cpp`](demos/death_test.cpp) — był już uruchomiony w pierwszej komórce.

```cpp
TEST(SafeDivideDeathTest, AbortsOnDivisionByZero) {
    EXPECT_DEATH(safeDivide(1, 0), "division by zero");
}
```

## `static_assert` — test wykonywany przez kompilator

Funkcje `constexpr` można testować **w czasie kompilacji**: `static_assert(warunek)` przerywa kompilację,
gdy warunek jest fałszywy ([cppreference — `static_assert`](https://en.cppreference.com/w/cpp/language/static_assert)).
Taki test nie wymaga uruchamiania programu ani frameworka — jeśli plik się kompiluje, test przeszedł
(zobacz [`demos/compile_time_test.cpp`](demos/compile_time_test.cpp)).

```cpp
constexpr int digitSum(int number) {
    int sum = 0;
    while (number > 0) {
        sum += number % 10;
        number /= 10;
    }
    return sum;
}

static_assert(digitSum(1234) == 10);
static_assert(digitSum(1234) == 11, "this one fails: the file does not compile");
```

Ciekawostka: w wyrażeniu stałym kompilator **nie dopuszcza zachowania niezdefiniowanego** — np.
przepełnienie `int` czy dzielenie przez zero podczas obliczeń `constexpr` to błąd kompilacji
([cppreference — constant expressions](https://en.cppreference.com/w/cpp/language/constant_expression)).
Test przez `static_assert` jest więc przy okazji darmowym detektorem UB dla sprawdzanych przypadków.
Typy i koncepty też można tak testować, np. `static_assert(std::is_nothrow_move_constructible_v<Widget>);`.

## Fuzzing

Testy jednostkowe sprawdzają przypadki wymyślone przez programistę. **Fuzzing** generuje ogromne liczby
losowych lub zmodyfikowanych danych wejściowych, kierując się pokryciem kodu (dane, które docierają do
nowych gałęzi, są dalej mutowane), i szuka awarii — najczęściej w połączeniu z sanitizerami
([Wikipedia](https://pl.wikipedia.org/wiki/Fuzzing)).

* [libFuzzer](https://llvm.org/docs/LibFuzzer.html) (Clang) — funkcja `LLVMFuzzerTestOneInput(const uint8_t*, size_t)`;
  autorzy przenieśli rozwój do nowszych narzędzi, ale błędy są nadal poprawiane,
* [FuzzTest](https://github.com/google/fuzztest) (Google) — testy fuzzingowe pisane obok zwykłych testów
  GoogleTest; działa na Linuksie z Clangiem:

```cpp
void ReversingTwiceGivesOriginal(const std::string& text) {
    EXPECT_EQ(reversed(reversed(text)), text);
}
FUZZ_TEST(ReversedTest, ReversingTwiceGivesOriginal);
```

* [AFL++](https://github.com/AFLplusplus/AFLplusplus) — inny popularny fuzzer,
* [OSS-Fuzz](https://google.github.io/oss-fuzz/) — ciągły fuzzing tysięcy projektów open source.

Wykład: [K. Serebryany — *Fuzz or lose...*, CppCon 2017](https://www.youtube.com/watch?v=k-Cv8Q3zWNQ).

## Testy oparte na właściwościach

Zamiast przykładów („`reversed("abc") == "cba"`”) zapisujemy **właściwość**, która musi zachodzić dla
*każdego* wejścia („odwrócenie dwa razy daje oryginał”). Framework losuje setki wejść, a gdy znajdzie
kontrprzykład, **zmniejsza go** (*shrinking*) do możliwie najprostszego — zamiast długiego, losowego wejścia
dostajemy najmniejsze, które wciąż pokazuje istotę błędu.

[RapidCheck](https://github.com/emil-e/rapidcheck) integruje się z GoogleTest makrem `RC_GTEST_PROP`
([dokumentacja](https://github.com/emil-e/rapidcheck/blob/master/doc/gtest.md)).
Demonstracja [`demos/property_test.cpp`](demos/property_test.cpp) sprawdza celowo błędną funkcję
`maxElement`, która zaczyna szukanie maksimum od 0 zamiast od pierwszego elementu. Testy
z „typowymi” danymi, np. `{3, 7, 5}`, tego nie wykryją. Zapisujemy dwie właściwości:

```cpp
RC_GTEST_PROP(MaxElementTest, ResultIsOneOfTheValues, (const std::vector<int>& values)) {
    RC_PRE(!values.empty());  // precondition: skip empty vectors
    const int result = maxElement(values);
    RC_ASSERT(std::find(values.begin(), values.end(), result) != values.end());
}

RC_GTEST_PROP(MaxElementTest, ResultIsNotSmallerThanAnyValue, (const std::vector<int>& values)) {
    const int result = maxElement(values);
    for (int value : values) {
        RC_ASSERT(result >= value);
    }
}
```

Komórka poniżej pobiera RapidCheck (wymaga internetu) i uruchamia oba testy. Zobacz, po ilu losowaniach
RapidCheck znalazł kontrprzykład i do jakiego wektora go zmniejszył. Zauważ też, że druga właściwość
przechodzi — sama nie wykryłaby błędu. Dobór właściwości jest tak samo ważny jak dobór przykładów.

In [ ]:
# Property-based testing demo (downloads RapidCheck on first run); the test fails on purpose
!cmake -S demos -B demos/build-rc -DENABLE_RAPIDCHECK=ON && cmake --build demos/build-rc --config Debug --parallel && ctest --test-dir demos/build-rc -C Debug -R MaxElement --output-on-failure

FuzzTest w trybie testów jednostkowych też działa jak testy oparte na właściwościach. Historia i motywacja
RapidCheck: [Spotify Engineering — *Generating test cases so you don't have to*](https://engineering.atspotify.com/2015/6/rapid-check).

## Kata TDD: Bowling Game

*Kata* to krótkie ćwiczenie powtarzane wielokrotnie, żeby wyrobić nawyk. Klasyczne kata TDD to
liczenie wyniku w kręglach (10 ramek, strike, spare, dodatkowe rzuty w ostatniej ramce) —
[opis zasad](https://kata-log.rocks/bowling-game-kata),
[wersja R. C. Martina krok po kroku](http://butunclebob.com/ArticleS.UncleBob.TheBowlingGameKata).
Spróbuj rozwiązać je w GoogleTest ściśle według cyklu Red – Green – Refactor z notebooka 5:
zacznij od listy przypadków (same zera, same jedynki, jeden spare, jeden strike, gra idealna = 300).

## Inne frameworki testowe

GoogleTest nie jest jedynym wyborem w C++:

* [Catch2](https://github.com/catchorg/Catch2) — nazwy testów to dowolne napisy, asercje to zwykłe wyrażenia
  C++ (`REQUIRE(a == b)` i tak wypisuje obie wartości), sekcje zamiast fixture'ów, proste makra BDD
  i mikrobenchmarki ([*Why Catch2?*](https://github.com/catchorg/Catch2/blob/devel/docs/why-catch.md)):

```cpp
#include <catch2/catch_test_macros.hpp>

TEST_CASE("reversing a word", "[reversed]") {
    REQUIRE(reversed("abc") == "cba");
}
```

* [doctest](https://github.com/doctest/doctest) — API podobne do Catch2, nastawione na bardzo szybką
  kompilację; testy mogą mieszkać w kodzie produkcyjnym,
* [Boost.Test](https://www.boost.org/doc/libs/release/libs/test/) — część Boost.

Wybór frameworka jest mniej ważny niż zasady z notebooków 0–5 — są takie same w każdym z nich.

## Linki i materiały

* [Clang — AddressSanitizer](https://clang.llvm.org/docs/AddressSanitizer.html), [UBSan](https://clang.llvm.org/docs/UndefinedBehaviorSanitizer.html)
* [GoogleTest — Death Tests](https://google.github.io/googletest/advanced.html#death-tests)
* [FuzzTest — CMake quickstart](https://github.com/google/fuzztest/blob/main/doc/quickstart-cmake.md)
* [RapidCheck + GoogleTest](https://github.com/emil-e/rapidcheck/blob/master/doc/gtest.md)
* [Kata-Log — Bowling Game](https://kata-log.rocks/bowling-game-kata)
* [P. Nash — *Back to Basics: Testing in C++*, CppCon 2023 (wideo)](https://www.youtube.com/watch?v=7_H4qzhWbnQ)